# BT-DKGRec-GCN — chạy thực nghiệm trên Google Colab

Đề án thạc sĩ: *Xây dựng ứng dụng dự báo hành vi khách hàng sử dụng đồ thị tri thức động*

**Phân công hạ tầng**

| | |
|---|---|
| VPS | Viết code, cấu trúc dự án, notebook, dựng KG cho demo, đẩy lên GitHub |
| **Colab (notebook này)** | **Chạy toàn bộ thực nghiệm — mọi mô hình, mọi seed, mọi cohort** |

**Chạy tuần tự từ trên xuống.** Mỗi phần in ra bằng chứng kiểm chứng chứ không chỉ báo "xong".

⚠️ Không sửa siêu tham số trực tiếp trong notebook này. Mọi tham số nằm trong `configs/`;
sửa ở đó, commit, rồi `git pull` lại — nếu không, kết quả sẽ không tái lập được từ repo.


## 1. Kiểm tra môi trường

In ra GPU, RAM và phiên bản Python để ghi vào phần "môi trường thực nghiệm" của luận văn.

In [ ]:
import subprocess, sys, platform

print("Python :", sys.version.split()[0], "|", platform.platform())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "khong co GPU")
except FileNotFoundError:
    print("khong co nvidia-smi — dang chay CPU")

with open("/proc/meminfo") as f:
    total = int(f.readline().split()[1]) / 1e6
print(f"RAM    : {total:.1f} GB")
print("\nGhi chu: cac moc san (popularity) khong can GPU, nhung cac mo hinh GCN o Buoc 6-8 thi can.")

## 2. Kết nối Google Drive

Drive dùng cho hai việc:
- **Đọc** dữ liệu RetailRocket thô (1,3 GB — quá lớn để đưa vào git)
- **Ghi** run artifact, để không mất khi Colab hết phiên

Cấu trúc Drive mong đợi:

```
MyDrive/BT-DKGRec/
├── raw/                    ← 4 file CSV của RetailRocket
│   ├── events.csv
│   ├── item_properties_part1.csv
│   ├── item_properties_part2.csv
│   └── category_tree.csv
└── runs/                   ← notebook sẽ tự tạo, chứa kết quả
```

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/BT-DKGRec")
DRIVE_RAW  = DRIVE_ROOT / "raw"
DRIVE_RUNS = DRIVE_ROOT / "runs"
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print("Drive:", DRIVE_ROOT)
if DRIVE_RAW.exists():
    for f in sorted(DRIVE_RAW.iterdir()):
        print(f"  {f.name:<32}{f.stat().st_size/1e6:>9.1f} MB")
else:
    print("  CHUA CO thu muc raw/ — xem o buoc 4 de tai du lieu ve")

## 3. Lấy mã nguồn từ GitHub

⚠️ **Điền `REPO_URL` trước khi chạy.** Repo chỉ chứa mã nguồn — dữ liệu và kết quả đều bị
`.gitignore` chặn, nên clone rất nhẹ.

Notebook luôn `pull` bản mới nhất để kết quả gắn với đúng một commit ghi lại được.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = ""          # <-- DIEN URL GitHub cua repo vao day
BRANCH   = "main"
REPO_DIR = Path("/content/bt-dkgrec")

assert REPO_URL, "Chua dien REPO_URL — xem o tren"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Dang chay tren commit:", commit)
print("\nGhi lai commit nay vao bang ket qua — do la thu gan so lieu voi ma nguon.")

## 4. Cài thư viện

`requirements-train.txt` = thư viện lõi + PyTorch. Colab đã có sẵn phần lớn nên bước này thường nhanh.

In [ ]:
!pip install -q -r requirements-train.txt

import importlib
for name in ("numpy", "pandas", "scipy", "pyarrow", "pydantic", "torch"):
    try:
        m = importlib.import_module(name)
        print(f"  {name:<10}{getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  {name:<10}THIEU")

import torch
print(f"\nCUDA kha dung: {torch.cuda.is_available()}",
      f"| {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "")

## 5. Đưa dữ liệu thô vào chỗ

`data/raw` là symlink trỏ sang Drive — **không copy 1,3 GB** vào máy Colab.

Nếu Drive chưa có dữ liệu, bỏ comment khối Kaggle bên dưới (cần `kaggle.json`).

In [ ]:
from pathlib import Path

RAW_FILES = ["events.csv", "item_properties_part1.csv",
             "item_properties_part2.csv", "category_tree.csv"]

# --- Neu Drive chua co du lieu, tai tu Kaggle (can kaggle.json trong Drive) ---
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && cp "{DRIVE_ROOT}/kaggle.json" ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d retailrocket/ecommerce-dataset -p /tmp/rr --unzip
# !mkdir -p "{DRIVE_RAW}" && cp /tmp/rr/*.csv "{DRIVE_RAW}/"

data_raw = Path("data/raw")
if data_raw.is_symlink() or data_raw.exists():
    data_raw.unlink() if data_raw.is_symlink() else None
data_raw.parent.mkdir(parents=True, exist_ok=True)
data_raw.symlink_to(DRIVE_RAW)

missing = [f for f in RAW_FILES if not (data_raw / f).exists()]
assert not missing, f"Thieu file raw: {missing}"
for f in RAW_FILES:
    print(f"  OK  {f:<32}{(data_raw / f).stat().st_size/1e6:>9.1f} MB")

## 6. Kiểm tra khung dự án

Chạy đúng bộ kiểm tra của Bước 1 và toàn bộ test. **Test đỏ thì dừng lại**, đừng chạy tiếp —
kết quả sinh ra từ code chưa xanh không có giá trị.

In [ ]:
!python scripts/00_check_setup.py
print("\n" + "=" * 70 + "\n")
!python -m pytest -q

## 7. Tiền xử lý — cả hai cohort

In bảng audit đối chiếu với luận văn v11. Mốc cứng nhất: **train events = 2.024.042** cho
cohort original. Lệch nhiều nghĩa là sai file raw hoặc sai cách đọc → dừng, không chạy tiếp.

Guard chống rò rỉ chạy tự động hai lượt trong mỗi lần preprocess (trong bộ nhớ, rồi đọc lại
từ Parquet). Không có cách nào tắt guard.

In [ ]:
%%time
!python scripts/01_preprocess.py --cohort original
print("\n" + "#" * 78 + "\n")
!python scripts/01_preprocess.py --cohort active

## 8. Dựng đồ thị tri thức — cả ba biến thể

`lightgcn` (không side info) · `static_kg_gcn` (có side info, trọng số đều) ·
`bt_dkgrec` (có side info, trọng số hành vi-thời gian).

Ba biến thể được dựng trong **cùng một lần chạy từ cùng một tập interim**, để không ai
chất vấn được rằng chúng đến từ dữ liệu khác nhau.

In [ ]:
%%time
!python scripts/02_build_graph.py --cohort original --all
print("\n" + "#" * 78 + "\n")
!python scripts/02_build_graph.py --cohort active --all

In [ ]:
# Kiem chung: bt_dkgrec va static_kg_gcn phai giong het nhau ve CAU TRUC,
# chi khac TRONG SO. Day la co so cua cau hoi hoi dong so 3 (DONG > TINH).
import json
import numpy as np
import scipy.sparse as sp

for cohort in ("original", "active"):
    a = sp.load_npz(f"data/processed/{cohort}/bt_dkgrec/adjacency.npz").tocsr()
    b = sp.load_npz(f"data/processed/{cohort}/static_kg_gcn/adjacency.npz").tocsr()
    sa = json.load(open(f"data/processed/{cohort}/bt_dkgrec/graph_stats.json"))
    sb = json.load(open(f"data/processed/{cohort}/static_kg_gcn/graph_stats.json"))
    print(f"[{cohort}]")
    print("  cung sparsity pattern :", np.array_equal(a.indptr, b.indptr) and np.array_equal(a.indices, b.indices))
    print("  cung so canh tung loai:", sa["edges"] == sb["edges"])
    print("  trong so KHAC nhau    :", not np.allclose(a.data, b.data))
    print("  khac dung mot thu     :", sa["weighting"], "vs", sb["weighting"])

## 9. Chạy mô hình

Mọi mô hình chạy trên **cùng bộ seed** `[2020, 2021, 2022]` (`experiments/seeds.json`) —
không có ngoại lệ, kể cả những mô hình tất định.

`popularity` và `recent_popularity` **tất định**: cả ba seed sẽ cho kết quả **giống hệt
nhau**, nên độ lệch chuẩn bằng 0. Đó là tính chất của mô hình, không phải lỗi tính toán —
phải chú thích điều này dưới bảng kết quả trong luận văn.

Colab nhiều RAM nên nâng `--eval-batch-size` để chạy nhanh hơn (VPS phải để 16).

In [ ]:
%%time
import json, subprocess, time

SEEDS = json.load(open("experiments/seeds.json"))["seeds"]
MODELS = ["popularity", "recent_popularity"]
COHORTS = ["original", "active"]
EVAL_BATCH = 512          # Colab co ~12 GB RAM; VPS chi chay duoc 16

failed = []
for cohort in COHORTS:
    for model in MODELS:
        for seed in SEEDS:
            tag = f"{cohort}/{model}/{seed}"
            t0 = time.time()
            proc = subprocess.run(
                ["python", "scripts/03_train.py", "--model", model, "--cohort", cohort,
                 "--seed", str(seed), "--eval-batch-size", str(EVAL_BATCH)],
                capture_output=True, text=True,
            )
            if proc.returncode == 0:
                print(f"  OK    {tag:<40}{time.time()-t0:>6.0f}s")
            else:
                print(f"  FAIL  {tag:<40}{time.time()-t0:>6.0f}s")
                print(proc.stdout[-1500:], proc.stderr[-1500:])
                failed.append(tag)

print(f"\n{len(COHORTS)*len(MODELS)*len(SEEDS) - len(failed)}"
      f"/{len(COHORTS)*len(MODELS)*len(SEEDS)} run thanh cong")
assert not failed, f"Co run that bai: {failed}"

## 10. Kiểm tra run artifact

Mỗi run phải có đủ 6 file. `curves.csv` là bắt buộc — khi bảo vệ cần chứng minh baseline đã hội tụ chứ không bị dừng sớm.

In [ ]:
from pathlib import Path

REQUIRED = ["config.yaml", "seed.txt", "metrics.json", "topk.csv", "curves.csv", "train.log"]
runs = sorted(p for p in Path("experiments/runs").iterdir() if p.is_dir())

print(f"{len(runs)} run\n")
for run in runs:
    missing = [f for f in REQUIRED if not (run / f).exists()]
    print(f"  {'OK  ' if not missing else 'THIEU'} {run.name:<52}{'' if not missing else missing}")

## 11. Bảng kết quả

Đọc **toàn bộ** run trong `experiments/runs/` — không có tham số lọc seed.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

K = 20
rows = []
for run in sorted(Path("experiments/runs").glob("*/metrics.json")):
    m = json.load(open(run))
    warm = m["test"]["warm"]
    if warm is None:
        continue
    rows.append({
        "cohort": m["cohort"], "model": m["model"], "seed": m["seed"],
        **{k: warm[f"{k}@{K}"] for k in ("recall", "ndcg", "hit_rate", "coverage")},
    })

frame = pd.DataFrame(rows)
summary = frame.groupby(["cohort", "model"]).agg(["mean", "std", "count"])
pd.set_option("display.width", 200, "display.float_format", lambda x: f"{x:.6f}")
print(f"TEST — phan doan WARM — K={K}   (mean +/- std tren {frame['seed'].nunique()} seed)\n")
print(summary)
print("\nstd = 0 o popularity/recent_popularity la do MO HINH TAT DINH, khong phai loi.")

## 12. Sao lưu kết quả về Drive

Colab xoá sạch máy khi hết phiên. Chạy ô này **trước khi đóng notebook**, nếu không mất hết run.

Chỉ sao lưu `experiments/runs/` — dữ liệu trung gian và đồ thị đều dựng lại được từ raw.

In [ ]:
import shutil, time
from pathlib import Path

stamp = time.strftime("%Y%m%d-%H%M%S")
target = DRIVE_RUNS / stamp
target.mkdir(parents=True, exist_ok=True)

n = 0
for run in sorted(p for p in Path("experiments/runs").iterdir() if p.is_dir()):
    shutil.copytree(run, target / run.name, dirs_exist_ok=True)
    n += 1

# Ghi kem commit dang chay de sau nay truy nguoc duoc so lieu ve ma nguon
(target / "COMMIT.txt").write_text(commit + "\n")
print(f"Da sao luu {n} run vao {target}")
print("Commit:", commit)